# Mapping migration

## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -d yearly_avg elpaso_Temp_F elpaso_df plot_yearly_avg
%store -r

Unable to restore variable 'gbif_ecoregion_gdf', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'elpaso_Temp_F', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'elpaso_df', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'plot_yearly_avg', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>


You will also need to import any libraries you are using in this notebook:

In [2]:
### import libraries
import os 
import pathlib
import geopandas as gpd
import urllib.request
import zipfile
import ee 
import pandas as pd 
import earthpy as et

## Step 4: Connect the GBIF observations to the ecoregions

### Step 4A: Identify the ecoregion for each observation

You can combine the ecoregions and the observations **spatially** using a method called `.sjoin()`, which stands for spatial join.

Check out the [`geopandas` documentation on spatial joins](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html#spatial-joins) to learn more about spatial joins and how they work.

To perform your spatial join, you'll need to identify the correct values for the `how =` and `predicate =` parameters of the spatial join.

In [3]:
### join the gbif observations to the ecoregion polygons
gbif_ecoregion_gdf = (
    #ecoregions_gdf
    gdf
    
    ### match the CRS of the gbif data and the ecoregions
    .to_crs(gbif_gdf.crs)

    ### find ecoregion for each observation
    .sjoin(
        gbif_gdf.rename_geometry('point_geometry'), #added .rename geom, this was not in there in orginial run that led to infinite loop for simplify
        how = 'inner', 
        predicate = None)

    ### any additional manipulations you want to do 
    ### (renaming columns, etc)
    .rename(columns= {
        'decimalLatitude': 'lat',
        'decimalLongitude': 'lon',
    })
)

### check it out
gbif_ecoregion_gdf

,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,...,kingdom,phylum,class,order,species,occurrenceID,individualCount,occurrenceStatus,lat,lon
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1745012658,2.0,PRESENT,50.518837,-116.044760
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1782800802,1.0,PRESENT,50.524776,-116.032970
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1781997715,1.0,PRESENT,50.524776,-116.032970
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1758799182,1.0,PRESENT,50.621094,-116.099785
12,13.0,Alberta-British Columbia foothills forests,5.0,Temperate Conifer Forests,Nearctic,NE05,2,345,62.333821,17.133639,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1764405743,1.0,PRESENT,50.519592,-122.492230
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1838044519,1.0,PRESENT,60.404890,-1.117499
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1838968253,1.0,PRESENT,60.404890,-1.117499
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS2949011428,1.0,PRESENT,60.404890,-1.117499
839,845.0,North Atlantic moist mixed forests,4.0,Temperate Broadleaf & Mixed Forests,Palearctic,PA04,3,672,89.144126,5.586107,...,Animalia,Chordata,Aves,Passeriformes,Catharus fuscescens,URN:catalog:CLO:EBIRD:OBS1838069664,1.0,PRESENT,60.404890,-1.117499


### Step 4B: Count the observations in each ecoregion each month

First, group the GBIF observations by ecoregion:
1. Replace `columns_to_group_by` with a list of columns. Keep in mind that you will end up with one row for each group – you want to count the observations in each ecoregion by month.
2. Select only month/ecoregion combinations that have more than one occurrence recorded, since a single occurrence could be an error.
3. Use the `.groupby()` and `.mean()` methods to compute the mean occurrences by ecoregion and by month.
4. Run the code – it will normalize the number of occurrences by month and ecoregion.

In [4]:
### summarize the occurrences by ecoregion/month combination
occurrence_df = (
    gbif_ecoregion_gdf

    ### group by each ecoregion and month 
    .groupby(['ECO_NAME', 'month'])

    ### and count the number of occurrences
    .agg(occurrences=('individualCount', 'sum'))
)

### get rid of rare observations (which are possibly misidentifications)
occurrence_df = occurrence_df[occurrence_df['occurrences'] > 1]

### calculate the mean # observations by ecoregion
mean_occurrences_by_ecoregion = (
    occurrence_df
    .groupby('ECO_NAME')
    .mean()
    
)

### calculate the mean # observations by month
mean_occurrences_by_month = (
    occurrence_df
    .groupby('month')
    .mean()
)

In [5]:
mean_occurrences_by_ecoregion

,occurrences
ECO_NAME,
Alberta-British Columbia foothills forests,233.750000
Aleutian Islands tundra,295.750000
Allegheny Highlands forests,4434.714286
Alto Paraná Atlantic forests,20.750000
Amazon-Orinoco-Southern Caribbean mangroves,39.666667
...,...
Western shortgrass prairie,155.600000
Wyoming Basin shrub steppe,50.500000
Xingu-Tocantins-Araguaia moist forests,10.666667


In [6]:
mean_occurrences_by_month.head()

,occurrences
month,
1,20.416667
2,10.777778
3,8.777778
4,461.181818
5,6899.925000


### Step 4C: Normalize the observations

Divide occurrences by the mean occurrences by month and by ecoregion.

In [7]:
### normalize by space and time for sampling effort
occurrence_df['norm_occurrences_ecoregion'] = (
    occurrence_df['occurrences']
    / mean_occurrences_by_ecoregion['occurrences']
)

occurrence_df['norm_occurrences_month'] = (
    occurrence_df['occurrences']
    / mean_occurrences_by_month['occurrences']
)
occurrence_df['norm_occurrences'] = (
    occurrence_df['occurrences']
    / mean_occurrences_by_ecoregion['occurrences']
    / mean_occurrences_by_month['occurrences']
)
### check it out
occurrence_df

occurrences  \
ECO_NAME                                   month                
Alberta-British Columbia foothills forests 5            193.0   
                                           6            431.0   
                                           7            296.0   
                                           8             15.0   
Aleutian Islands tundra                    5            164.0   
...                                                       ...   
Yucatán dry forests                        9              5.0   
Yucatán moist forests                      4             14.0   
                                           5              9.0   
                                           9             17.0   
                                           10             7.0   

                                                  norm_occurrences_ecoregion  \
ECO_NAME                                   month                               
Alberta-British Columbia foothills forests 5                        0.825668   
                                           6                        1.843850   
                                           7                        1.266310   
                                           8                        0.064171   
Aleutian Islands tundra                    5                        0.554522   
...                                                                      ...   
Yucatán dry forests                        9                        0.652174   
Yucatán moist forests                      4                        1.191489   
                                           5                        0.765957   
                                           9                        1.446809   
                                           10                       0.595745   

                                                  norm_occurrences_month  \
ECO_NAME                                   month                           
Alberta-British Columbia foothills forests 5                    0.027971   
                                           6                    0.052421   
                                           7                    0.078122   
                                           8                    0.017101   
Aleutian Islands tundra                    5                    0.023768   
...                                                                  ...   
Yucatán dry forests                        9                    0.004080   
Yucatán moist forests                      4                    0.030357   
                                           5                    0.001304   
                                           9                    0.013871   
                                           10                   0.066675   

                                                  norm_occurrences  
ECO_NAME                                   month                    
Alberta-British Columbia foothills forests 5              0.000120  
                                           6              0.000224  
                                           7              0.000334  
                                           8              0.000073  
Aleutian Islands tundra                    5              0.000080  
...                                                            ...  
Yucatán dry forests                        9              0.000532  
Yucatán moist forests                      4              0.002584  
                                           5              0.000111  
                                           9              0.001181  
                                           10             0.005674  

[504 rows x 4 columns]

## Wrap up Step 4

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [8]:
%store occurrence_df mean_occurrences_by_ecoregion mean_occurrences_by_month

Stored 'occurrence_df' (DataFrame)
Stored 'mean_occurrences_by_ecoregion' (DataFrame)
Stored 'mean_occurrences_by_month' (DataFrame)


Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!